# ===== Scenario: 5-class =====

In [ ]:
from utils import *
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
tf.config.experimental.set_memory_growth(tf.config.list_physical_devices("GPU")[0], True)

(x_train, y_train, x_val, y_val, x_test, y_test, _, _, _) = load_jetnet(
    mode="5class",
    n_train_pool=780000,
    n_test_pool=100000,
    n_train=700000,
    n_val=50000,
    n_test=95000,
    seed=42
)

#plot_jets(x=x_train, y=y_train, idx=0)
#plot_jets_aug(x=x_train, y=y_train, mask_ratio_range=(0.3, 0.3), idx=3, save_path="figs/jets.png")

### pretraining

In [ ]:
d_model=32; n_heads=4; n_layers=2; d_proj=16
#d_model=64; n_heads=6; n_layers=4; d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")

proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

teacher.set_weights(student.get_weights())
proj_head_t.set_weights(proj_head_s.get_weights())
teacher.trainable = False
proj_head_t.trainable = False

masker = MaskTokens(d_model, name="particle_masker")
center_cls = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_cls")
center_patch = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_patch")

#print(student.summary())
#print(proj_head_s.summary())

batch_size = 1024
base_lr = 5e-4 * (batch_size / 256)

history = train_jbot(
    x_train=x_train,
    epochs=80,
    batch_size=batch_size,
    optimizer=keras.optimizers.AdamW(learning_rate=base_lr, weight_decay=1e-4),
    base_lr=base_lr,
    warmup_epochs=10,
    ema_tau=0.996,
    student=student,
    teacher=teacher,
    n_layers=n_layers,
    d_proj=d_proj,
    proj_head_s=proj_head_s,
    proj_head_t=proj_head_t,
    mask_prob=1,
    mask_ratio_range=(0, 0.5),
    masker=masker,
    center_cls=center_cls,
    center_patch=center_patch,
    center_beta=0.9,
    temp_t=0.04,
    temp_s=0.1,
    use_hint=False,
    hint_hidden=0,
    lambda_koleo=0.01,
    save_student_snapshots=True,
    snapshot_dir=f"snapshots_all_{d_model}_{n_heads}_{n_layers}_v1")

plot_jbot_pretraining(history)

plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_train, y_sample=y_train)
plot_softmax_prob_cls(n_samples=5000, x=x_train, student=student, teacher=teacher, proj_head_s=proj_head_s, proj_head_t=proj_head_t, center_cls=center_cls, temp_t=0.04, temp_s=0.1)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test, alpha=0.5, marker_size=10)
#plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4, alpha=0.5, marker_size=10)
pretrain_probe_5class(backbone=student, x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test, knn_n=20, k_folds=10)

In [ ]:
student.save_weights(f"models/all_backbone_pretrain_{d_model}_{n_heads}_{n_layers}_v1.weights.h5")
history_json(history, f"models/all_history_pretrain_{d_model}_{n_heads}_{n_layers}_v1.json", mode="save")

In [ ]:
plot_jbot_pretraining(history_json(None, "models/all_history_pretrain_64_6_4.json", mode="load"), save_path="figs/all_history_pretrain_64_6_4.png")

In [ ]:
plot_sne_snapshots("snapshots_all_32_4_2", x_test, y_test, save=True)

### downstream

In [ ]:
d_model=32; n_heads=4; n_layers=2; d_proj=16
#d_model=64; n_heads=6; n_layers=4; d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
student.load_weights(f"models/all_backbone_pretrain_{d_model}_{n_heads}_{n_layers}.weights.h5")

plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_test, y_sample=y_test, save_path=f"figs/all_attention_pretrain_{d_model}_{n_heads}_{n_layers}.png")
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test, alpha=0.5, marker_size=10)
#plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4, alpha=0.5, marker_size=10)
pretrain_probe_5class(backbone=student, x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test, knn_n=20, k_folds=10)

In [ ]:
epochs = 200
batch_size = 1024
lr = 1e-4 * (batch_size / 256)
tolerance = 1e-4
patience = 20

# small: base_lr=1e-04  decay=0.65
# base:  base_lr=5e-05  decay=0.70

backbone_standalone, mlp_standalone, model_standalone, history_standalone = train_standalone(
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    n_classes=5,
    lr=lr,
    epochs=epochs,
    batch_size=batch_size,
    tolerance=tolerance,
    patience=patience
)

backbone_ft, mlp_ft, model_ft, history_ft = finetune(
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    n_classes=5,
    backbone_pretrain=student,
    base_lr=lr,
    decay=0.65,
    epochs=epochs,
    batch_size=batch_size,
    tolerance=tolerance,
    patience=patience
)

plot_training_histories(history_standalone, history_ft, labels=("supervised", "jBOT"))

In [ ]:
y_pred_standalone = model_standalone.predict(x_test)
y_pred_ft = model_ft.predict(x_test)

report_acc_eff(y_test, {"supervised": y_pred_standalone, "jBOT": y_pred_ft}, ("q","g","W","Z","t"), (0.1, 0.01, 0.001), 10)
plot_roc_ft_5class(y_test, y_pred_standalone, y_pred_ft, save_path=f"figs/all_roc_ft_{d_model}_{n_heads}_{n_layers}.png")
plot_tSNE_cls(n_samples=5000, backbone=backbone_ft, x=x_test, y=y_test, alpha=0.5, marker_size=10, save_path=f"figs/all_tsne_ft_{d_model}_{n_heads}_{n_layers}.png")
#plot_pca_corner_cls_embeddings(backbone=backbone_ft, x=x_test, y=y_test, n_samples=5000, n_components=4, alpha=0.5, marker_size=10)

In [ ]:
backbone_standalone.save_weights(f"models/all_backbone_standalone_{d_model}_{n_heads}_{n_layers}.weights.h5")
mlp_standalone.save_weights(f"models/all_mlp_standalone_{d_model}_{n_heads}_{n_layers}.weights.h5")
backbone_ft.save_weights(f"models/all_backbone_ft_{d_model}_{n_heads}_{n_layers}.weights.h5")
mlp_ft.save_weights(f"models/all_mlp_ft_{d_model}_{n_heads}_{n_layers}.weights.h5")

In [ ]:
d_model=32; n_heads=4; n_layers=2; d_proj=16
#d_model=64; n_heads=6; n_layers=4; d_proj=32

_, _, model_standalone = load_finetune(f"all_backbone_standalone_{d_model}_{n_heads}_{n_layers}", f"all_mlp_standalone_{d_model}_{n_heads}_{n_layers}", d_model, n_heads, n_layers, 5)
backbone_ft, _, model_ft = load_finetune(f"all_backbone_ft_{d_model}_{n_heads}_{n_layers}", f"all_mlp_ft_{d_model}_{n_heads}_{n_layers}", d_model, n_heads, n_layers, 5)

### fine-tuning scan

In [ ]:
d_model=32; n_heads=4; n_layers=2; d_proj=16

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
student.load_weights(f"models/all_backbone_pretrain_{d_model}_{n_heads}_{n_layers}.weights.h5")

scan_finetune_lr_decay(
    base_lrs=[1e-5, 2e-5, 5e-5, 1e-4, 2e-4, 5e-4],
    decays=[0.6, 0.65, 0.7, 0.75, 0.8],
    train_frac=0.6,
    x_train=x_train, y_train=y_train,
    x_val=x_val, y_val=y_val,
    x_test=x_test, y_test=y_test,
    d_model=d_model, n_heads=n_heads, n_layers=n_layers, n_classes=5,
    backbone_pretrain=student,
    epochs=200, batch_size=1024, tolerance=1e-4, patience=20,
)

In [ ]:
d_model=64; n_heads=6; n_layers=4; d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
student.load_weights(f"models/all_backbone_pretrain_{d_model}_{n_heads}_{n_layers}.weights.h5")

scan_finetune_lr_decay(
    base_lrs=[1e-5, 2e-5, 5e-5, 1e-4, 2e-4, 5e-4],
    decays=[0.6, 0.65, 0.7, 0.75, 0.8],
    train_frac=0.6,
    x_train=x_train, y_train=y_train,
    x_val=x_val, y_val=y_val,
    x_test=x_test, y_test=y_test,
    d_model=d_model, n_heads=n_heads, n_layers=n_layers, n_classes=5,
    backbone_pretrain=student,
    epochs=200, batch_size=1024, tolerance=1e-4, patience=20,
)

# ===== Scenario: top vs qg =====

In [ ]:
from utils import *
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
tf.config.experimental.set_memory_growth(tf.config.list_physical_devices("GPU")[0], True)

(x_train, y_train, x_val, y_val, x_test, y_test, y_train_top, y_val_top, y_test_top) = load_jetnet(
    mode="tqg",
    n_train_pool=755000,
    n_test_pool=125000,
    n_train=285000,
    n_val=20000,
    n_test=50000,
    seed=42
)

y_train_2 = np.eye(2, dtype=np.float32)[y_train_top]
y_val_2 = np.eye(2, dtype=np.float32)[y_val_top]
y_test_2 = np.eye(2, dtype=np.float32)[y_test_top]

### pretraining

In [ ]:
d_model=32; n_heads=4; n_layers=2; d_proj=16
#d_model=64; n_heads=6; n_layers=4; d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")

proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

teacher.set_weights(student.get_weights())
proj_head_t.set_weights(proj_head_s.get_weights())
teacher.trainable = False
proj_head_t.trainable = False

masker = MaskTokens(d_model, name="particle_masker")
center_cls = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_cls")
center_patch = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_patch")

#print(student.summary())
#print(proj_head_s.summary())

batch_size = 1024
base_lr = 5e-4 * (batch_size / 256)

history = train_jbot(
    x_train=x_train,
    epochs=80,
    batch_size=batch_size,
    optimizer=keras.optimizers.AdamW(learning_rate=base_lr, weight_decay=1e-4),
    base_lr=base_lr,
    warmup_epochs=10,
    ema_tau=0.996,
    student=student,
    teacher=teacher,
    n_layers=n_layers,
    d_proj=d_proj,
    proj_head_s=proj_head_s,
    proj_head_t=proj_head_t,
    mask_prob=1,
    mask_ratio_range=(0, 0.5),
    masker=masker,
    center_cls=center_cls,
    center_patch=center_patch,
    center_beta=0.9,
    temp_t=0.04,
    temp_s=0.1,
    use_hint=False,
    hint_hidden=0,
    lambda_koleo=0.01,
    save_student_snapshots=True,
    snapshot_dir=f"snapshots_tqg_{d_model}_{n_heads}_{n_layers}_v1")

plot_jbot_pretraining(history)

plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_train, y_sample=y_train)
plot_softmax_prob_cls(n_samples=5000, x=x_train, student=student, teacher=teacher, proj_head_s=proj_head_s, proj_head_t=proj_head_t, center_cls=center_cls, temp_t=0.04, temp_s=0.1)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test, alpha=0.5, marker_size=10)
#plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4, alpha=0.5, marker_size=10)
pretrain_probe_tqg(backbone=student, x_train=x_train, y_train_top=y_train_top, x_test=x_test, y_test_top=y_test_top, y_test_2=y_test_2, knn_n=20)

In [ ]:
student.save_weights(f"models/tqg_backbone_pretrain_{d_model}_{n_heads}_{n_layers}_v1.weights.h5")
#history_json(history, f"models/tqg_history_pretrain_{d_model}_{n_heads}_{n_layers}_v1.json", mode="save")

In [ ]:
plot_jbot_pretraining(history_json(None, "models/tqg_history_pretrain_64_6_4.json", mode="load"))

In [ ]:
plot_sne_snapshots("snapshots_tqg_64_6_4", x_test, y_test, save=True)

### downstream

In [ ]:
#d_model=32; n_heads=4; n_layers=2; d_proj=16
d_model=64; n_heads=6; n_layers=4; d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
student.load_weights(f"models/tqg_backbone_pretrain_{d_model}_{n_heads}_{n_layers}_v1.weights.h5")

plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_test, y_sample=y_test)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test, alpha=0.5, marker_size=10)
#plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4, alpha=0.5, marker_size=10)
pretrain_probe_tqg(backbone=student, x_train=x_train, y_train_top=y_train_top, x_test=x_test, y_test_top=y_test_top, y_test_2=y_test_2, knn_n=20)

In [ ]:
epochs = 200
batch_size = 1024
lr = 1e-4 * (batch_size / 256)
tolerance = 1e-4
patience = 20

# small: base_lr=1e-04  decay=0.80
# base:  base_lr=1e-04  decay=0.70

backbone_standalone, mlp_standalone, model_standalone, history_standalone = train_standalone(
    x_train=x_train,
    y_train=y_train_2,
    x_val=x_val,
    y_val=y_val_2,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    n_classes=2,
    lr=lr,
    epochs=epochs,
    batch_size=batch_size,
    tolerance=tolerance,
    patience=patience
)

backbone_ft, mlp_ft, model_ft, history_ft = finetune(
    x_train=x_train,
    y_train=y_train_2,
    x_val=x_val,
    y_val=y_val_2,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    n_classes=2,
    backbone_pretrain=student,
    base_lr=lr,
    decay=0.7,
    epochs=epochs,
    batch_size=batch_size,
    tolerance=tolerance,
    patience=patience
)

plot_training_histories(history_standalone, history_ft, labels=("supervised", "jBOT"))

In [ ]:
y_pred_standalone = model_standalone.predict(x_test)
y_pred_ft = model_ft.predict(x_test)

report_acc_eff(y_test_2, {"supervised": y_pred_standalone, "jBOT": y_pred_ft}, ("QCD","t"), (0.1, 0.01, 0.001), 10)
plot_roc_ft_tqg(y_test_2, y_pred_standalone, y_pred_ft, save_path=f"figs/tqg_roc_ft_{d_model}_{n_heads}_{n_layers}.png")
plot_tSNE_cls(n_samples=5000, backbone=backbone_ft, x=x_test, y=y_test, alpha=0.5, marker_size=10, save_path=f"figs/tqg_tsne_ft_{d_model}_{n_heads}_{n_layers}.png")
#plot_pca_corner_cls_embeddings(backbone=backbone_ft, x=x_test, y=y_test, n_samples=5000, n_components=4, alpha=0.5, marker_size=10)

In [ ]:
backbone_standalone.save_weights(f"models/tqg_backbone_standalone_{d_model}_{n_heads}_{n_layers}.weights.h5")
mlp_standalone.save_weights(f"models/tqg_mlp_standalone_{d_model}_{n_heads}_{n_layers}.weights.h5")
backbone_ft.save_weights(f"models/tqg_backbone_ft_{d_model}_{n_heads}_{n_layers}.weights.h5")
mlp_ft.save_weights(f"models/tqg_mlp_ft_{d_model}_{n_heads}_{n_layers}.weights.h5")

In [ ]:
#d_model=32; n_heads=4; n_layers=2; d_proj=16
d_model=64; n_heads=6; n_layers=4; d_proj=32

_, _, model_standalone = load_finetune(f"tqg_backbone_standalone_{d_model}_{n_heads}_{n_layers}", f"tqg_mlp_standalone_{d_model}_{n_heads}_{n_layers}", d_model, n_heads, n_layers, 2)
backbone_ft, _, model_ft = load_finetune(f"tqg_backbone_ft_{d_model}_{n_heads}_{n_layers}", f"tqg_mlp_ft_{d_model}_{n_heads}_{n_layers}", d_model, n_heads, n_layers, 2)

### fine-tuning scan

In [ ]:
d_model=32; n_heads=4; n_layers=2; d_proj=16

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
student.load_weights(f"models/tqg_backbone_pretrain_{d_model}_{n_heads}_{n_layers}.weights.h5")

scan_finetune_lr_decay(
    base_lrs=[1e-5, 2e-5, 5e-5, 1e-4, 2e-4, 5e-4],
    decays=[0.6, 0.65, 0.7, 0.75, 0.8],
    train_frac=0.8,
    x_train=x_train, y_train=y_train_2,
    x_val=x_val, y_val=y_val_2,
    x_test=x_test, y_test=y_test_2,
    d_model=d_model, n_heads=n_heads, n_layers=n_layers, n_classes=2,
    backbone_pretrain=student,
    epochs=200, batch_size=1024, tolerance=1e-4, patience=20,
)

In [ ]:
d_model=64; n_heads=6; n_layers=4; d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
student.load_weights(f"models/tqg_backbone_pretrain_{d_model}_{n_heads}_{n_layers}.weights.h5")

scan_finetune_lr_decay(
    base_lrs=[1e-5, 2e-5, 5e-5, 1e-4, 2e-4, 5e-4],
    decays=[0.6, 0.65, 0.7, 0.75, 0.8],
    train_frac=0.8,
    x_train=x_train, y_train=y_train_2,
    x_val=x_val, y_val=y_val_2,
    x_test=x_test, y_test=y_test_2,
    d_model=d_model, n_heads=n_heads, n_layers=n_layers, n_classes=2,
    backbone_pretrain=student,
    epochs=200, batch_size=1024, tolerance=1e-4, patience=20,
)

# ===== Scenario: anomaly detection =====

In [ ]:
from utils import *
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
tf.config.experimental.set_memory_growth(tf.config.list_physical_devices("GPU")[0], True)

(x_train, y_train, _, _, x_test, y_test, _, _, _) = load_jetnet(
    mode="qg",
    n_train=260000,
    qg_balance_anoms=True,
    seed=42
)

### pretraining

In [ ]:
d_model=32; n_heads=4; n_layers=2; d_proj=16
#d_model=64; n_heads=6; n_layers=4; d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")

proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

teacher.set_weights(student.get_weights())
proj_head_t.set_weights(proj_head_s.get_weights())
teacher.trainable = False
proj_head_t.trainable = False

masker = MaskTokens(d_model, name="particle_masker")
center_cls = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_cls")
center_patch = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_patch")

#print(student.summary())
#print(proj_head_s.summary())

batch_size = 1024
base_lr = 5e-4 * (batch_size / 256)

history = train_jbot(
    x_train=x_train,
    epochs=60,
    batch_size=batch_size,
    optimizer=keras.optimizers.AdamW(learning_rate=base_lr, weight_decay=1e-4),
    base_lr=base_lr,
    warmup_epochs=10,
    ema_tau=0.996,
    student=student,
    teacher=teacher,
    n_layers=n_layers,
    d_proj=d_proj,
    proj_head_s=proj_head_s,
    proj_head_t=proj_head_t,
    mask_prob=1,
    mask_ratio_range=(0, 0.5),
    masker=masker,
    center_cls=center_cls,
    center_patch=center_patch,
    center_beta=0.9,
    temp_t=0.04,
    temp_s=0.1,
    use_hint=False,
    hint_hidden=0,
    lambda_koleo=0.01,
    save_student_snapshots=True,
    snapshot_dir=f"snapshots_qg_{d_model}_{n_heads}_{n_layers}_v2")

plot_jbot_pretraining(history)

plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_test, y_sample=y_test)
plot_softmax_prob_cls(n_samples=5000, x=x_train, student=student, teacher=teacher, proj_head_s=proj_head_s, proj_head_t=proj_head_t, center_cls=center_cls, temp_t=0.04, temp_s=0.1)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test, alpha=0.5, marker_size=10)
#plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4, alpha=0.5, marker_size=10)

In [ ]:
student.save_weights(f"models/qg_backbone_pretrain_{d_model}_{n_heads}_{n_layers}_v2.weights.h5")
#history_json(history, f"models/qg_history_pretrain_{d_model}_{n_heads}_{n_layers}_v1.json", mode="save")

In [ ]:
plot_jbot_pretraining(history_json(None, "models/qg_history_pretrain_64_6_4.json", mode="load"))

In [ ]:
plot_sne_snapshots("snapshots_qg_64_6_4", x_test, y_test, save=True)

### downstream

In [ ]:
d_model=32; n_heads=4; n_layers=2; d_proj=16
#d_model=64; n_heads=6; n_layers=4; d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
student.load_weights(f"models/qg_backbone_pretrain_{d_model}_{n_heads}_{n_layers}.weights.h5")

plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_test, y_sample=y_test)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test, alpha=0.5, marker_size=10)
#plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4, alpha=0.5, marker_size=10)

z_train = get_cls_embeddings(student, x_train)
z_test = get_cls_embeddings(student, x_test)

In [ ]:
scan_maha = {
    "name": "maha",
    "M": [20000],
    "l2": [True],
    "reg": [1e-5, 1e-8]
}

scan_knn = {
    "name": "knnL2",
    "M": [20000],
    "l2": [True],
    "kmax": [2000],
    "k": [15, 20, 25, 30, 35, 40, 55, 60, 100],
    "agg": ["mean"] # mean, median, max
}

scan_cos = {
    "name": "cos",
    "M": [20000],
    "kmax": [2000],
    "k": [20, 25, 30, 35, 40, 45, 50, 55, 60, 100],
    "agg": ["logsumexp"], # logsumexp, softmax_mean, max
    "temp": [0.02, 0.04, 0.05]
}

scan_ccmd = {
    "name": "ccMD(min_qg)",
    "M": [20000],
    "l2": [True],
    "reg": [1e-5, 1e-8]
}

scan_rmd = {
    "name": "RMD(min_qg)",
    "M": [20000],
    "l2": [True],
    "reg": [1e-5, 1e-8],
    "variant": ["sub"]
}

scan_gmm = {
    "name": "GMM",
    "M": [20000],
    "l2": [True],
    "K": [1, 2, 4, 8],
    "cov_type": ["full", "tied"],
    "reg_covar": [1e-5, 1e-8],
    "n_init": [1],
    "seed": 123
}

run_ad_scan(
    z_train, z_test, y_test,
    seed=123,
    fpr_targets=(1e-1, 1e-2, 1e-3),
    scan_maha=scan_maha,
    scan_knn=scan_knn,
    scan_cos=scan_cos,
    scan_ccmd=scan_ccmd,
    scan_rmd=scan_rmd,
    scan_gmm=scan_gmm,
    y_train5=y_train,
    highlight_top=20
)

In [ ]:
seed_scan = 123
bank = make_bank(z_train, M=20000, seed=seed_scan)

d_knn = knn_distances(bank, z_test, l2norm=True, kmax=2000)
score_knn = knn_aggregate(d_knn, k=30, agg="mean")
    
sims_cos = cosine_similarities(bank, z_test, kmax=2000)
score_cos = cosine_aggregate(sims_cos, k=100, agg="logsumexp",temp=0.02)

z_bank_ccmd, y_bank_ccmd = make_bank_with_labels(z_train, y_train, M=20000, seed=seed_scan)
score_ccmd = cc_mahalanobis_min_qg_score(z_bank_ccmd, y_bank_ccmd, z_test, l2norm=True, reg=1e-8)

score_gmm = gmm_score(bank, z_test, K=4, cov_type="full", l2norm=True, reg_covar=1e-5, seed=seed_scan)

In [ ]:
plot_ad_score_hist(score_knn, y_test, title="$k$-NN", xlim=[min(score_knn),0.8], save_path=f"figs/score_knn.png")
plot_ad_score_hist(score_cos, y_test, title="Cosine similarity", xlim=[-1,-0.75], save_path=f"figs/score_cos.png")
plot_ad_score_hist(score_ccmd, y_test, title="Mahalanobis distance", xlim=[0,150], save_path=f"figs/score_ccmd.png")
plot_ad_score_hist(score_gmm, y_test, title="Gaussian mixture model", xlim=[min(score_gmm),20], save_path=f"figs/score_gmm.png")
    
plot_ad_roc_mult_scores_one_signal(
    {"$k$-NN": score_knn, "Cosine": score_cos, "Maha.": score_ccmd, "GMM": score_gmm},
    y_test, signal="t", kfold=10, save_path=f"figs/ad_roc_t.png"
)
plot_ad_roc_mult_scores_one_signal(
    {"$k$-NN": score_knn, "Cosine": score_cos, "Maha.": score_ccmd, "GMM": score_gmm},
    y_test, signal="W", kfold=10, save_path=f"figs/ad_roc_w.png"
)
plot_ad_roc_mult_scores_one_signal(
    {"$k$-NN": score_knn, "Cosine": score_cos, "Maha.": score_ccmd, "GMM": score_gmm},
    y_test, signal="Z", kfold=10, save_path=f"figs/ad_roc_z.png"
)
plot_ad_roc_mult_scores_one_signal(
    {"$k$-NN": score_knn, "Cosine": score_cos, "Maha.": score_ccmd, "GMM": score_gmm},
    y_test, signal="combined", kfold=10, save_path=f"figs/ad_roc_combined.png"
)

report_ad_kfold({"kNN": score_knn, "Cosine": score_cos, "Maha": score_ccmd, "GMM": score_gmm}, y_test, kfold=10, eff_b_targets=(0.1, 0.01, 0.001), seed=42)